# Projeto Final – Storytelling sobre Desastres Naturais (2014—2021)

## Setup

### Imports

In [11]:
import os
import pandas as pd
import plotly.express as px
from typing import List
from dotenv import load_dotenv
from mysql.connector import connect

### Environment Variables

In [12]:
load_dotenv()

MYSQL_HOST=os.getenv(key="MYSQL_HOST")
MYSQL_PORT=os.getenv(key="MYSQL_PORT")
MYSQL_USER=os.getenv(key="MYSQL_USER")
MYSQL_PASS=os.getenv(key="MYSQL_PASS")
MYSQL_DB=os.getenv(key="MYSQL_DB")

### MySQL Connector

In [39]:
connector = connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASS,
    database=MYSQL_DB,
    ssl_disabled=False
)

## Plot Charts

Q1 – Quais tipos de desastre predominaram entre 2014 e 2021?

In [20]:
q1_query = """
    SELECT
        cobrade AS desastre,
        COUNT(*) AS ocorrencias
    FROM fato_desastre
    WHERE status = 'Reconhecido'
    GROUP BY cobrade
    ORDER BY ocorrencias DESC
    LIMIT 10;
"""


cursor = connector.cursor(dictionary=True)
cursor.execute(q1_query)
rows = cursor.fetchall()


q1_items: List[dict] = []
for row in rows:
    temp: List[str] = row.get("desastre", "").split('-')  # type: ignore (Pylance)

    if len(temp) > 2 and ',' not in temp[2]:
        disaster = temp[2]
    else:
        disaster = temp[1]

    q1_items.append({
        "disaster": disaster,
        "occurrences": row.get("ocorrencias", "")  # type: ignore (Pylance)
    })


q1_df = pd.DataFrame(q1_items)
q1_df.head(n=10)

,disaster,occurrences
0,Doenças infecciosas virais,12790
1,Estiagem,9742
2,Seca,4507
3,Chuvas Intensas,1586
4,Enxurradas,801
5,Inundações,649
6,Vendaval,444
7,Granizo,257
8,Incêndio Florestal,237
9,Alagamentos,136


Criar gráfico

In [21]:
fig = px.bar(
    q1_df.sort_values("occurrences"),
    x="occurrences",
    y="disaster",
    orientation="h",
    text="occurrences",
    color="occurrences",
    color_continuous_scale="Teal"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    title="Desastres Predominantes entre 2014 e 2021",
    title_x=0.5,
    template="plotly_white",
    xaxis_title="Ocorrências",
    yaxis_title="",
    showlegend=False,
    coloraxis_colorbar=dict(title="")
)

fig.show()

Q2 – Onde os desastres se concentraram geograficamente?

In [24]:
q2_query = """
    SELECT
        uf,
        municipio,
        COUNT(*) AS ocorrencias
    FROM fato_desastre
    WHERE status = 'Reconhecido'
    GROUP BY uf, municipio
    ORDER BY ocorrencias DESC
    LIMIT 20;
"""


cursor = connector.cursor(dictionary=True)
cursor.execute(q2_query)
rows = cursor.fetchall()


q2_items = [
    {
        "state": row.get("uf", ""),                 # type: ignore (Pylance)
        "city": row.get("municipio", ""),           # type: ignore (Pylance)
        "occurrences": row.get("ocorrencias", "")   # type: ignore (Pylance)
    }
    for row in rows
]


q2_df = pd.DataFrame(q2_items)
q2_df.head(n=20)

,state,city,occurrences
0,MG,Itaobim,23
1,PE,Caruaru,22
2,PB,Itaporanga,21
3,MG,Capitão Enéas,21
4,MG,Janaúba,21
5,MG,Bocaiúva,21
6,MG,Rio Pardo de Minas,21
7,MG,Manga,21
8,MG,Almenara,21
9,MG,Juramento,21


Criar gráfico
- Bolha

In [27]:

fig = px.scatter(
    q2_df.sort_values("occurrences", ascending=False),
    x="state",
    y="city",
    size="occurrences",
    color="occurrences",
    hover_name="city",
    size_max=60,
    color_continuous_scale="Teal"
)

fig.update_layout(
    title="Maior Concentração Geográfica de Desastres",
    title_x=0.5,
    template="plotly_white",
    xaxis_title="Estado",
    yaxis_title="Cidade",
    coloraxis_colorbar=dict(title="")
)

fig.update_traces(
    marker=dict(
        sizemode='area',
        sizeref=2.*q2_df["occurrences"].max()/(60**2),
        sizemin=6
    )
)

fig.show()

Criar gráfico
- Treemap

In [25]:
fig = px.treemap(
    q2_df,
    path=["state", "city"],
    values="occurrences",
    color="occurrences",
    color_continuous_scale="Blues"
)

fig.update_layout(
    title="Maior Concentração Geográfica de Desastres",
    title_x=0.5,
    template="plotly_white",
    coloraxis_colorbar=dict(title="")
)

fig.show()

Q3 – Quais tipos de desastre geraram maior impacto humano entre 2014 e 2019 (**sem COVID**)

In [ ]:
q3_query = """
    SELECT
        cobrade,
        uf,
        municipio,
        SUM(dh_mortos) AS total_mortes
    FROM fato_desastre fd
    WHERE status = 'Reconhecido' AND ano BETWEEN 2014 and 2019 -- sem COVID
    GROUP BY cobrade, uf, municipio
    ORDER BY total_mortes DESC
    LIMIT 10;
"""


cursor = connector.cursor(dictionary=True)
cursor.execute(q3_query)
rows = cursor.fetchall()


q3_items = [
    {
        "disaster": row.get("cobrade", "").split('-')[1],   # type: ignore (Pylance)
        "state": row.get("uf", ""),                         # type: ignore (Pylance)
        "city": row.get("municipio", ""),                   # type: ignore (Pylance)
        "deaths": row.get("total_mortes", "")               # type: ignore (Pylance)
    }
    for row in rows
]


q3_df = pd.DataFrame(q3_items)
q3_df.head(n=10)

,disaster,state,city,deaths
0,Rompimento/colapso de barragens,MG,Brumadinho,171
1,Estiagem,ES,Barra de São Francisco,20
2,Deslizamentos,BA,Salvador,15
3,Inundações,SP,Itaóca,12
4,Doenças infecciosas virais,MG,Ladainha,11
5,Tempestade Local/Convectiva,RJ,Rio de Janeiro,10
6,Inundações,AM,Parintins,8
7,Rompimento/colapso de barragens,MG,Mariana,7
8,Estiagem,BA,Nordestina,7
9,Tempestade Local/Convectiva,PE,Camaragibe,7


Criar gráfico
- Treemap

In [37]:
fig = px.treemap(
    q3_df,
    path=["disaster", "state", "city"],
    values="deaths",
    color="deaths",
    color_continuous_scale="Reds"
)

fig.update_traces(
    texttemplate="%{label}<br>%{value} mortes"
)

fig.update_layout(
    title="Tipos de Desastre com Maior Impacto Humano (2014—2019)",
    title_x=0.5,
    template="plotly_white",
    coloraxis_colorbar=dict(title="")
)

fig.show()

Criar gráfico
- Heatmap

In [34]:
df_heat = q3_df.groupby(["state", "disaster"], as_index=False)["deaths"].sum()

fig = px.density_heatmap(
    df_heat,
    x="state",
    y="disaster",
    z="deaths",
    color_continuous_scale="Reds"
)

fig.update_layout(
    title="Tipos de Desastre com Maior Impacto Humano (2014—2019)",
    title_x=0.5,
    template="plotly_white"
)

fig.show()

Q3 – Quais tipos de desastre geraram maior impacto humano entre 2014 e 2019 (**com COVID**)

In [40]:
q3_covid_query = """
    SELECT
        cobrade,
        uf,
        municipio,
        SUM(dh_mortos) AS total_mortes
    FROM fato_desastre fd
    WHERE status = 'Reconhecido'
    GROUP BY cobrade, uf, municipio
    ORDER BY total_mortes DESC
    LIMIT 10;
"""


cursor = connector.cursor(dictionary=True)
cursor.execute(q3_covid_query)
rows = cursor.fetchall()


q3_covid_items = [
    {
        "disaster": row.get("cobrade", "").split('-')[1],   # type: ignore (Pylance)
        "state": row.get("uf", ""),                         # type: ignore (Pylance)
        "city": row.get("municipio", ""),                   # type: ignore (Pylance)
        "deaths": row.get("total_mortes", "")               # type: ignore (Pylance)
    }
    for row in rows
]


q3_covid_df = pd.DataFrame(q3_covid_items)
q3_covid_df.head(n=10)

,disaster,state,city,deaths
0,Doenças infecciosas virais,RJ,Rio de Janeiro,44519
1,Doenças infecciosas virais,RN,Natal,5506
2,Doenças infecciosas virais,DF,Brasília,4498
3,Doenças infecciosas virais,AP,Macapá,1814
4,Doenças infecciosas virais,PA,Bragança,1488
5,Doenças infecciosas virais,RN,Mossoró,1247
6,Doenças infecciosas virais,RJ,Nova Iguaçu,1247
7,Doenças infecciosas virais,PR,Foz do Iguaçu,1026
8,Doenças infecciosas virais,SP,Bauru,686
9,Doenças infecciosas virais,SP,São Vicente,668


Criar gráfico
- Treemap

In [49]:
fig = px.treemap(
    q3_covid_df,
    path=["disaster", "state", "city"],
    values="deaths",
    color="deaths",
    color_continuous_scale="Reds"
)

fig.update_traces(
    texttemplate="%{label}<br>%{value} mortes"
)

fig.update_layout(
    title="Tipos de Desastre com Maior Impacto Humano (2014—2021)",
    title_x=0.5,
    template="plotly_white",
    coloraxis_colorbar=dict(title="")
)

fig.show()

Criar gráfico
- Heatmap

In [51]:
df_heat = q3_covid_df.groupby(
    ["state", "disaster"],
    as_index=False
)["deaths"].sum()

fig = px.imshow(
    df_heat.pivot(index="disaster", columns="state", values="deaths"),
    text_auto=True,
    color_continuous_scale="Reds"
)

fig.update_layout(
    title="Tipos de Desastre com Maior Impacto Humano (2014—2021)",
    title_x=0.5,
    xaxis_title="Estado",
    yaxis_title="Desastre",
    template="plotly_white"
)

fig.show()